In [9]:
import numpy as np
import cvxpy as cp
import mosek
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [10]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_1(x,n):
    y = 1-(1-x)**n
    return(y)

def h_2(x,r):
    y = (1+r)*x-r*x**2
    return(y)

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def h_4(x,r):
    if x < 1/2:
        return((1+r)*x)
    else:
        return((1-r)*p+r)

In [11]:
def Robust_exp_h1 (x,p,r,m):
    N = len(x)
    q = cp.Variable(N)
    q_b = cp.Variable(N)
    phi_cons = 0
    psets = list(powerset(list(range(N))))
    for i in range(1,len(psets)):
        psets[i] = list(psets[i])
    psets = psets[1:(len(psets)-1)]
    for i in range(N):
        phi_cons = phi_cons -(cp.entr(q[i]) + q[i]*np.log(p[i]))
    constraints = [q >= 0, q_b >= 0, cp.sum(q) == 1, cp.sum(q_b) == 1, phi_cons <= r]
    for i in range(len(psets)):
        z1 = q[psets[i]]
        z2 = q_b[psets[i]]
        constraints.append(cp.sum(z2)-(1-(1-cp.sum(z1))**m) <= 0)
    obj = cp.Minimize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,q.value,q_b.value)


def Robust_dual_h1 (x, p, r, m):
    N = len(x)
    lbda = cp.Variable(N)
    v = cp.Variable((N,N))
    alpha = cp.Variable(1)
    gamma = cp.Variable(1)
    c = 1/(m**(1/(m-1)))+1/(m**(m/(m-1)))
    z2 = 0
    constraints = [gamma >= 0, lbda >= 0]
    for i in range(N):
        for j in range((N-i)):
            constraints.append(v[i,j] <= 0)
        constraints.append(-x[i]-cp.sum(lbda[(N-i):N]))
    for j in range(N):
        z1 = -cp.min(v[j,(N+1-j):N])
        z2 = z2 + z1 + c*z1**(m/(m-1))*1/(lbda[j]**(1/(m-1))) + lbda[j]
    v = 1/(alpha)**(1/3)
    print(v.curvature)
    print(z2.curvature)
    return(0) 

In [12]:
def Robust_exp_h3 (x,p,r,m):
    N = len(x)
    q = cp.Variable(N)
    q_b = cp.Variable(N)
    phi_cons = 0
    psets = list(powerset(list(range(N))))
    for i in range(1,len(psets)):
        psets[i] = list(psets[i])
    psets = psets[1:(len(psets)-1)]
    for i in range(N):
        phi_cons = phi_cons -(cp.entr(q[i]) + q[i]*np.log(p[i]))
    constraints = [q >= 0, q_b >= 0, cp.sum(q) == 1, cp.sum(q_b) == 1, phi_cons <= r]
    for i in range(len(psets)):
        z1 = q[psets[i]]
        z2 = q_b[psets[i]]
        v= -cp.neg(cp.sum(z1)/(1-m)-1)+1
        constraints.append(cp.sum(z2)-v <= 0)
    obj = cp.Maximize(-q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,q.value,q_b.value)  




In [84]:
def Robust_portfolio_h3_pos (x, p, r, r_f, m, c):
    N = len(x)
    lbda = cp.Variable(N)
    v = cp.Variable((N,N))
    t = cp.Variable(N, nonneg = True)
    alpha = cp.Variable(1)
    gamma = cp.Variable(1, nonneg = True)
    a = cp.Variable(1, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N-1):
        constraints.append(v[i][0:(N-i-1)] <= 0)
        constraints.append(v[i][(N-i-1):N] >= 0)
        constraints.append(-a*x[i]-cp.sum(lbda[(N-i-1):N]) <= 0)
    constraints.append(-a*x[N-1]-cp.sum(lbda) <= 0)
    constraints.append(v[N-1] >= 0)
    for j in range(N):
        z1 = -cp.min(v[j,(N-1-j):N])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    for i in range(N):
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:N:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0) 
    constraints.append(alpha + gamma * r - (1-a)*r_f + z4 -1 + z2 <= c)
    #constraints.append(a <= 1)
    obj = cp.Maximize(a*x.T @ p + (1-a)*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,v.value,a.value) 

In [86]:
def Robust_portfolio_h3_neg (x, p, r, r_f, m, c):
    N = len(x)
    lbda = cp.Variable(N)
    v = cp.Variable((N,N))
    t = cp.Variable(N, nonneg = True)
    alpha = cp.Variable(1)
    gamma = cp.Variable(1, nonneg = True)
    a = cp.Variable(1, nonpos = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(1,N):
        constraints.append(v[i-1][i:N] <= 0)
        constraints.append(v[i-1][0:i] >= 0)
        constraints.append(-a*x[i-1]-cp.sum(lbda[(i-1):N]) <= 0)
    constraints.append(-a*x[N-1]-lbda[N-1] <= 0)
    constraints.append(v[N-1][0:N] >= 0)
    for j in range(N):
        z1 = -cp.min(v[j,(N-1-j):N])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    for i in range(N):
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:N:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0) 
    constraints.append(alpha + gamma * r - (1-a)*r_f + z4 -1 + z2 <= c)
    #constraints.append(a <= 1)
    obj = cp.Maximize(a*x.T @ p + (1-a)*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,v.value,a.value) 

In [151]:
x=np.array([0.13,0.06,-0.12,-0.2])
p=np.array([0.2,0.4,0.3,0.1])
r=0.21
m=0.5
c = 0.3
r_f = 0.001
print(Robust_portfolio_h3_pos(x,p,r,r_f,m,c))
print(p.dot(x))

(0.0009999999999996923, array([[-0.        , -0.        , -0.        ,  0.10178977],
       [-0.        , -0.        ,  0.10205415,  0.10205415],
       [-0.        ,  0.1074825 ,  0.1074825 ,  0.1074825 ],
       [ 0.12196537,  0.12196537,  0.12196537,  0.12196537]]), array([4.392872e-14]))
-0.005999999999999998


In [152]:
Robust_portfolio_h3_neg(x,p,r,r_f,m,c)

(0.07159688164481749,
 array([[ 3.54894776e-09, -6.04747348e-02, -8.52778404e-02,
         -3.13566382e-08],
        [-0.00000000e+00, -0.00000000e+00, -2.93978008e-08,
         -2.93978008e-08],
        [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
         -2.80867089e-08],
        [ 8.70972996e-09,  1.46364988e-09,  1.46364988e-09,
          1.41790826e-08]]),
 array([-10.08526881]))

In [6]:
N=4
x=np.array([1,2,3])
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets)-1)]
psets

[[0],
 [1],
 [2],
 [3],
 [0, 1],
 [0, 2],
 [0, 3],
 [1, 2],
 [1, 3],
 [2, 3],
 [0, 1, 2],
 [0, 1, 3],
 [0, 2, 3],
 [1, 2, 3]]

In [7]:
for i in range(len(psets)):
    print(sum(qbvalue[psets[i]])-h_3(sum(qvalue[psets[i]]),m))
print(qbvalue)

NameError: name 'qbvalue' is not defined

In [7]:
print(qbvalue)
print(qvalue)

[9.99574480e-01 4.10014239e-04 1.54870993e-05 1.39706767e-08]
[0.85637513 0.08091854 0.04707461 0.01563172]


In [62]:
x=np.array([[0,1,2,3],[7,8,9,6]])
x[0:2:1,1]

array([1, 8])

In [22]:
wa = [1,2,3,4]
wa[0:1]

[1]